In [6]:
!pip install pandas sqlalchemy pymysql

In [7]:
import pandas as pd
from sqlalchemy import create_engine

# Replace 'your_password' with your actual MySQL root password
# If you are not using the 'root' user, change that as well.
db_user = 'root'
db_password = 'password'
db_host = 'localhost'
db_name = 'bingeplay'

# Create the connection engine using SQLAlchemy and pymysql
engine = create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")

print("Connection established successfully!")

Connection established successfully!


## Q1 - Active Revenue
**Answer:** The output below provides the total active subscriptions and the Monthly Recurring Revenue (MRR) as of June 30, 2024.

In [9]:
query_q1 = """
SELECT COUNT(subscription_id) AS active_subscriptions,SUM(monthly_price_inr) AS total_monthly_revenue
FROM subscriptions WHERE status = 'active' AND (end_date IS NULL OR end_date > '2024-06-30');
"""
pd.read_sql(query_q1, engine)

,active_subscriptions,total_monthly_revenue
0,2340,784260.0


## Q2 - Signup Momentum
**Answer:** The table below shows the signups per month for H1 2024. The peak momentum month is the one with the highest `signup_count`.

In [10]:
query_q2 = """
SELECT 
    MONTH(signup_date) AS month_number,
    MONTHNAME(signup_date) AS month_name,
    COUNT(user_id) AS signup_count
FROM users
WHERE signup_date BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY month_number;
"""
pd.read_sql(query_q2, engine)

,month_number,month_name,signup_count
0,1,January,350
1,2,February,400
2,3,March,500
3,4,April,550
4,5,May,600
5,6,June,600


## Q3 - Device Analytics
**Answer:** The breakdown of engagement and completion rates across Mobile, TV, Laptop, and Tablet, excluding invalid guest sessions (`user_id IS NULL`).

In [11]:
query_q3 = """
SELECT 
    device_type,
    COUNT(session_id) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes_per_session,
    ROUND(100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(session_id), 2) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""
pd.read_sql(query_q3, engine)

,device_type,total_sessions,total_watch_minutes,avg_watch_minutes_per_session,completion_rate_pct
0,Mobile,50172,1504355.0,29.98,60.24
1,TV,27981,840595.0,30.04,59.98
2,Laptop,15105,453434.0,30.02,60.51
3,Tablet,7091,210733.0,29.72,59.79


## Q4 - Rating Distribution
**Answer:** The first output shows the exact distribution of star ratings. The second output shows the percentage of all ratings that are highly positive (4 or 5 stars).

In [12]:
query_q4a = """
SELECT 
    stars,
    COUNT(rating_id) AS rating_count,
    ROUND(100.0 * COUNT(rating_id) / (SELECT COUNT(*) FROM ratings), 2) AS rating_percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""
print("--- Star Distribution ---")
display(pd.read_sql(query_q4a, engine))

query_q4b = """
SELECT 
    ROUND(100.0 * SUM(CASE WHEN stars >= 4 THEN 1 ELSE 0 END) / COUNT(rating_id), 2) AS pct_4_or_5_stars
FROM ratings;
"""
print("\n--- High Satisfaction % ---")
display(pd.read_sql(query_q4b, engine))

--- Star Distribution ---


,stars,rating_count,rating_percentage
0,1,234,4.68
1,2,352,7.04
2,3,847,16.94
3,4,1781,35.62
4,5,1786,35.72



--- High Satisfaction % ---


,pct_4_or_5_stars
0,71.34


## Q5 - Originals vs Acquired
**Answer:** The table below compares catalog size, average release year, and average IMDb rating.
**Interpretation:** BingePlay Originals (is_original = 1) typically feature higher overall ratings compared to Acquired content, indicating that exclusive, internal content is driving stronger user satisfaction.

In [13]:
query_q5 = """
SELECT 
    CASE WHEN is_original = 1 THEN 'BingePlay Original' ELSE 'Acquired' END AS content_type,
    COUNT(show_id) AS total_shows,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 1) AS avg_release_year
FROM shows
GROUP BY is_original;
"""
pd.read_sql(query_q5, engine)

,content_type,total_shows,avg_imdb_rating,avg_release_year
0,Acquired,70,6.63,2020.7
1,BingePlay Original,30,7.92,2020.4


## Q6 - Binge Day Detection
**Answer:** • Total number of binge days that occurred (across all users and shows) 
• The user_id who had the MOST binge days in Q2 2024 
• How many binge days that user had

In [14]:
query_q6 = """
WITH Q2_Binge_Days AS (
    SELECT 
        user_id,
        show_id,
        session_date,
        COUNT(session_id) AS daily_sessions
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(session_id) >= 5
)
SELECT COUNT(*) AS total_q2_binge_days FROM Q2_Binge_Days;
"""
display(pd.read_sql(query_q6, engine))

query_q6b = """
WITH Q2_Binge_Days AS (
    SELECT user_id, show_id, session_date
    FROM watch_sessions
    WHERE user_id IS NOT NULL AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(session_id) >= 5
)
SELECT user_id, COUNT(*) AS max_binge_days
FROM Q2_Binge_Days
GROUP BY user_id
ORDER BY max_binge_days DESC
LIMIT 1;
"""
display(pd.read_sql(query_q6b, engine))

,total_q2_binge_days
0,414


,user_id,max_binge_days
0,U02956,8


## Q7 - Q1 Signups who never watched
**Deliverables:**
• Total number of Q1 signups 
• Number of Q1 signups who have never watched anything

In [15]:
query_q7 = """
SELECT 
    COUNT(u.user_id) AS total_q1_signups,
    COUNT(CASE WHEN ws.user_id IS NULL THEN 1 END) AS never_watched
FROM users u
LEFT JOIN (SELECT DISTINCT user_id FROM watch_sessions WHERE user_id IS NOT NULL) ws 
    ON u.user_id = ws.user_id
WHERE MONTH(u.signup_date) <= 3 
  AND YEAR(u.signup_date) = 2024;
"""
pd.read_sql(query_q7, engine)

,total_q1_signups,never_watched
0,1250,226


## Q8 - The over-paying Premium/Family users
**Deliverables:**
• Number of Premium/Family users whose entire watch history is Basic-tier only

In [18]:
query_q8 = """
WITH CurrentPremiumFamilyUsers AS (
    SELECT user_id FROM subscriptions
    WHERE status = 'active' 
      AND (end_date IS NULL OR end_date > '2024-06-30')
      AND plan IN ('Premium', 'Family')
)
SELECT COUNT(DISTINCT c.user_id) AS overpaying_users
FROM CurrentPremiumFamilyUsers c
WHERE NOT EXISTS (
    SELECT 1 FROM watch_sessions ws
    JOIN shows s ON ws.show_id = s.show_id
    WHERE ws.user_id = c.user_id
      AND s.min_plan IN ('Premium', 'Family')
);
"""
pd.read_sql(query_q8, engine)

,overpaying_users
0,212


## Q9 - Upgrade success cohort
**Deliverables:** 
• Number of such users 
• Their average days from signup to first upgrade 

In [19]:
query_q9 = """
WITH Ordered_Subscriptions AS (
    SELECT user_id, plan, start_date, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date ASC) AS sub_rank
    FROM subscriptions
),
January_Basic_Starts AS (
    SELECT u.user_id, u.signup_date, os.start_date AS basic_start_date
    FROM users u
    JOIN Ordered_Subscriptions os ON u.user_id = os.user_id
    WHERE YEAR(u.signup_date) = 2024 AND MONTH(u.signup_date) = 1
      AND os.sub_rank = 1 AND os.plan = 'Basic'
),
First_Upgrades AS (
    SELECT s.user_id, MIN(s.start_date) AS first_upgrade_date
    FROM subscriptions s
    JOIN January_Basic_Starts jbs ON s.user_id = jbs.user_id
    WHERE s.plan IN ('Premium', 'Family') AND s.start_date > jbs.basic_start_date
    GROUP BY s.user_id
)
SELECT 
    COUNT(DISTINCT jbs.user_id) AS total_upgraded_users,
    ROUND(AVG(DATEDIFF(up.first_upgrade_date, jbs.signup_date)), 2) AS avg_days_to_first_upgrade
FROM January_Basic_Starts jbs
JOIN First_Upgrades up ON jbs.user_id = up.user_id
JOIN subscriptions act ON jbs.user_id = act.user_id
WHERE act.status = 'active' AND act.start_date <= '2024-06-30' AND (act.end_date IS NULL OR act.end_date > '2024-06-30');
"""
pd.read_sql(query_q9, engine)

,total_upgraded_users,avg_days_to_first_upgrade
0,55,64.96


## Q10 - Cliffhanger comebacks


In [22]:
query_q10a = """
 SELECT s1.show_id, sh.title, COUNT(*) as comeback_events
FROM watch_sessions s1
JOIN watch_sessions s2 ON s1.user_id = s2.user_id AND s1.show_id = s2.show_id
JOIN shows sh ON s1.show_id = sh.show_id
WHERE s1.completed = 0 AND s1.user_id IS NOT NULL
AND s2.session_date BETWEEN DATE_ADD(s1.session_date, INTERVAL 1 DAY) 
AND DATE_ADD(s1.session_date, INTERVAL 7 DAY)
GROUP BY s1.show_id, sh.title
ORDER BY comeback_events DESC;

"""
display(pd.read_sql(query_q10a, engine))

,show_id,title,comeback_events
0,S005,Show Special 27,77
1,S088,Rayalaseema Raga,76
2,S024,Show Special 18,72
3,S019,Founder Diaries,70
4,S065,Show Special 21,68
...,...,...,...
95,S039,Show Special 0,36
96,S064,Madurai Masala,35
97,S066,Dil Dosti Duniya,34
98,S047,Filmy Files,34


## Q11 - Consecutive-week engagement
**Deliverables:** 
• Number of users with a streak of 4+ consecutive weeks 
• The longest streak in weeks (across all users) 
• One user_id who has that longest streak

In [23]:
query_q11a = """
WITH Weekly_Activity AS (
    SELECT DISTINCT 
        user_id, 
        WEEK(session_date, 3) + (YEAR(session_date) * 52) AS abs_week
    FROM watch_sessions WHERE user_id IS NOT NULL
),
Streaks AS (
    SELECT 
        user_id, 
        abs_week,
        (abs_week - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY abs_week)) AS streak_group
    FROM Weekly_Activity
),
Streak_Lengths AS (
    SELECT user_id, streak_group, COUNT(*) AS streak_weeks
    FROM Streaks GROUP BY user_id, streak_group
)
SELECT COUNT(DISTINCT user_id) AS users_with_4_plus_weeks
FROM Streak_Lengths
WHERE streak_weeks >= 4;
"""
display(pd.read_sql(query_q11a, engine))

query_q11b = """
WITH Weekly_Activity AS (
    SELECT DISTINCT user_id, WEEK(session_date, 3) + (YEAR(session_date) * 52) AS abs_week
    FROM watch_sessions WHERE user_id IS NOT NULL
),
Streaks AS (
    SELECT user_id, abs_week, (abs_week - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY abs_week)) AS streak_group
    FROM Weekly_Activity
),
Streak_Lengths AS (
    SELECT user_id, COUNT(*) AS streak_weeks FROM Streaks GROUP BY user_id, streak_group
)
SELECT user_id, MAX(streak_weeks) AS longest_streak_in_weeks
FROM Streak_Lengths
GROUP BY user_id
ORDER BY longest_streak_in_weeks DESC
LIMIT 1;
"""
display(pd.read_sql(query_q11b, engine))

,users_with_4_plus_weeks
0,1675


,user_id,longest_streak_in_weeks
0,U00213,26


## Q12 - Churn signal detection
Find users whose total watch_minutes in June 2024 dropped by 50% or more compared to their total 
watch_minutes in May 2024. These are early churn signals - the product team can target them with 
retention offers. 

In [24]:
query_q12 = """
WITH Monthly_Watch AS (
    SELECT 
        user_id,
        SUM(CASE WHEN session_date BETWEEN '2024-05-01' AND '2024-05-31' THEN watch_minutes ELSE 0 END) AS may_minutes,
        SUM(CASE WHEN session_date BETWEEN '2024-06-01' AND '2024-06-30' THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL AND session_date BETWEEN '2024-05-01' AND '2024-06-30'
    GROUP BY user_id
),
Churn_Risk_Users AS (
    SELECT 
        m.user_id, u.name, m.may_minutes, m.june_minutes,
        ROUND(100.0 * (m.may_minutes - m.june_minutes) / m.may_minutes, 2) AS drop_percentage
    FROM Monthly_Watch m
    JOIN users u ON m.user_id = u.user_id
    WHERE m.may_minutes > 0 AND ((m.may_minutes - m.june_minutes) / m.may_minutes) >= 0.50
)
SELECT COUNT(*) AS total_churn_signal_users FROM Churn_Risk_Users;
"""
display(pd.read_sql(query_q12, engine))

query_q12b = """
WITH Monthly_Watch AS (
    SELECT user_id,
        SUM(CASE WHEN session_date BETWEEN '2024-05-01' AND '2024-05-31' THEN watch_minutes ELSE 0 END) AS may_minutes,
        SUM(CASE WHEN session_date BETWEEN '2024-06-01' AND '2024-06-30' THEN watch_minutes ELSE 0 END) AS june_minutes
    FROM watch_sessions WHERE user_id IS NOT NULL AND session_date BETWEEN '2024-05-01' AND '2024-06-30' GROUP BY user_id
),
Churn_Risk_Users AS (
    SELECT m.user_id, u.name, m.may_minutes, m.june_minutes,
        ROUND(100.0 * (m.may_minutes - m.june_minutes) / m.may_minutes, 2) AS drop_percentage
    FROM Monthly_Watch m JOIN users u ON m.user_id = u.user_id
    WHERE m.may_minutes > 0 AND ((m.may_minutes - m.june_minutes) / m.may_minutes) >= 0.50
)
SELECT user_id, name, may_minutes, june_minutes, drop_percentage
FROM Churn_Risk_Users
ORDER BY drop_percentage DESC, may_minutes DESC;
"""
display(pd.read_sql(query_q12b, engine))

,total_churn_signal_users
0,521


,user_id,name,may_minutes,june_minutes,drop_percentage
0,U01503,Myra Patil,759.0,0.0,100.00
1,U01397,Shreya Shenoy,727.0,0.0,100.00
2,U00292,Kavitha Kamath,594.0,0.0,100.00
3,U02199,Advait Shah,559.0,0.0,100.00
4,U00883,Atharv Verma,547.0,0.0,100.00
...,...,...,...,...,...
516,U01858,Sanjay Mukherjee,296.0,147.0,50.34
517,U02530,Nandini Roy,451.0,224.0,50.33
518,U02100,Vikram Singh,204.0,102.0,50.00
519,U01192,Rohit Mukherjee,136.0,68.0,50.00
